# AseguraPy | Rubi, agente documental de seguros

Proyecto académico ficticio para el Challenge Alura Agente. Rubi responde exclusivamente con fragmentos recuperados de la Base Documental de AseguraPy y muestra las páginas utilizadas.

**Diseño:** PDF -> extracción por página -> fragmentos -> BM25 local -> Gemini 3.1 Flash-Lite (opcional) -> Gradio. Si Gemini no está disponible, la recuperación sigue funcionando sin inventar información.

## 1. Instalación

Ejecutá esta celda una sola vez en un entorno nuevo de Google Colab. Se usa BM25 porque es liviano, gratuito y compatible con ARM64 para el despliegue posterior en OCI.

In [ ]:
!pip -q install -U google-genai gradio>=6.0.0 pypdf rank-bm25

import sys, subprocess
print("Python:", sys.version.split()[0])
print("Dependencias instaladas.")

## 2. Clave API segura de Gemini

En Colab abrí el icono de llave lateral, creá el secreto **GEMINI_API_KEY**, pegá tu clave y activá el acceso para este notebook. Nunca pegues una clave dentro de una celda ni la subas a GitHub.

In [ ]:
import os
from google.colab import userdata

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None

if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    print("Clave de Gemini encontrada de forma segura.")
else:
    print("No se encontró GEMINI_API_KEY. El buscador BM25 funcionará; Gemini quedará desactivado.")

## 3. Configuración del modelo y diagnóstico

El modelo objetivo es `gemini-3.1-flash-lite`. La celda prueba una solicitud mínima, lista modelos si el identificador falla y explica los errores comunes: 401 (clave inválida), 404 (modelo no disponible), 429 (cuota agotada) y 400 (solicitud o disponibilidad regional).

In [ ]:
from google import genai
from google.genai import types

GEMINI_MODEL = "gemini-3.1-flash-lite"
client = None
MODELO_LISTO = False

def listar_modelos_disponibles():
    if client is None:
        return []
    try:
        return [m.name for m in client.models.list() if "generateContent" in getattr(m, "supported_generation_methods", [])]
    except Exception as error:
        print("No fue posible listar modelos:", type(error).__name__)
        return []

def explicar_error_api(error):
    texto = str(error)
    if "401" in texto or "UNAUTHENTICATED" in texto or "API key not valid" in texto:
        return "401: la clave API es inválida, no está activa o no se leyó desde Secretos de Colab."
    if "404" in texto or "NOT_FOUND" in texto:
        return "404: el modelo no está disponible para este proyecto o el nombre no es válido. Revisá la lista de modelos."
    if "429" in texto or "RESOURCE_EXHAUSTED" in texto:
        return "429: se agotó temporalmente la cuota o el límite de solicitudes. Esperá y reintentá."
    if "400" in texto or "FAILED_PRECONDITION" in texto:
        return "400: solicitud no válida o el nivel gratuito no está disponible para la configuración actual."
    return f"Error de Gemini: {type(error).__name__}. {texto[:300]}"

if GEMINI_API_KEY:
    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        prueba = client.models.generate_content(
            model=GEMINI_MODEL,
            contents="Respondé solamente: OK",
            config=types.GenerateContentConfig(max_output_tokens=10),
        )
        MODELO_LISTO = bool(getattr(prueba, "text", "").strip())
        print("Gemini disponible:", MODELO_LISTO)
    except Exception as error:
        print(explicar_error_api(error))
        disponibles = listar_modelos_disponibles()
        if disponibles:
            print("Modelos disponibles para este proyecto:")
            print("\n".join(disponibles[:30]))
else:
    print("Sin clave: se usará solamente recuperación BM25.")

## 4. Subir el PDF

Descargá `AseguraPy_Base_Documental_v1.0.pdf` desde el proyecto y seleccionálo cuando Colab abra el diálogo. No subas documentos personales.

In [ ]:
from google.colab import files
from pathlib import Path

archivos = files.upload()
if not archivos:
    raise ValueError("No se subió ningún archivo PDF.")

PDF_PATH = Path(next(iter(archivos.keys())))
if PDF_PATH.suffix.lower() != ".pdf":
    raise ValueError("El archivo debe tener extensión .pdf")
print(f"PDF cargado: {PDF_PATH.name} ({PDF_PATH.stat().st_size:,} bytes)")

## 5. Lectura por página y fragmentación

Cada fragmento conserva su página de origen. Esto permite mostrar fuentes y evitar enviar el documento completo al modelo.

In [ ]:
import re
from pypdf import PdfReader

def normalizar(texto):
    return re.sub(r"\s+", " ", (texto or "")).strip()

def extraer_paginas(pdf_path):
    lector = PdfReader(str(pdf_path))
    paginas = []
    for numero, pagina in enumerate(lector.pages, start=1):
        texto = normalizar(pagina.extract_text())
        if texto:
            paginas.append({"pagina": numero, "texto": texto})
    if not paginas:
        raise ValueError("El PDF no contiene texto extraíble. Usá un PDF con texto, no una imagen escaneada.")
    return paginas

def dividir_en_fragmentos(paginas, tamano=850, solapamiento=140):
    fragmentos = []
    for item in paginas:
        texto = item["texto"]
        inicio = 0
        while inicio < len(texto):
            fin = min(len(texto), inicio + tamano)
            if fin < len(texto):
                corte = texto.rfind(". ", inicio, fin)
                if corte > inicio + 250:
                    fin = corte + 1
            bloque = texto[inicio:fin].strip()
            if bloque:
                fragmentos.append({
                    "id": len(fragmentos),
                    "pagina": item["pagina"],
                    "texto": bloque,
                })
            if fin >= len(texto):
                break
            inicio = max(fin - solapamiento, inicio + 1)
    return fragmentos

paginas = extraer_paginas(PDF_PATH)
fragmentos = dividir_en_fragmentos(paginas)
print(f"Páginas con texto: {len(paginas)}")
print(f"Fragmentos creados: {len(fragmentos)}")
print("Ejemplo:", fragmentos[0]["texto"][:300])

## 6. Sistema de recuperación BM25

BM25 es la estrategia principal. No requiere una base vectorial, GPU, FAISS ni embeddings remotos; por eso sigue siendo adecuada para Colab gratuito y OCI Ampere A1 ARM64.

In [ ]:
from rank_bm25 import BM25Okapi

STOPWORDS = {"a", "al", "ante", "bajo", "con", "contra", "de", "del", "desde", "el", "en", "entre", "es", "la", "las", "lo", "los", "para", "por", "que", "se", "sin", "su", "un", "una", "y"}

def tokenizar(texto):
    return [t for t in re.findall(r"[a-záéíóúüñ0-9]+", texto.lower()) if t not in STOPWORDS]

corpus_tokens = [tokenizar(item["texto"]) for item in fragmentos]
bm25 = BM25Okapi(corpus_tokens)

def buscar_fragmentos(pregunta, k=4):
    consulta_tokens = tokenizar(pregunta)
    if not consulta_tokens:
        return []
    puntajes = bm25.get_scores(consulta_tokens)
    orden = sorted(range(len(puntajes)), key=lambda i: puntajes[i], reverse=True)
    resultados = [fragmentos[i] | {"puntaje": float(puntajes[i])} for i in orden[:k] if puntajes[i] > 0]
    return resultados

def formatear_fuentes(resultados):
    paginas_fuente = sorted({r["pagina"] for r in resultados})
    return ", ".join(f"página {n}" for n in paginas_fuente) if paginas_fuente else "sin fuentes"

print("Índice BM25 listo.")

## 7. Prueba de recuperación sin modelo

Esta prueba demuestra que la búsqueda documental funciona incluso sin clave API.

In [ ]:
pregunta_prueba = "¿Cuál es el plazo para denunciar un robo de vehículo?"
resultados_prueba = buscar_fragmentos(pregunta_prueba)
print("Pregunta:", pregunta_prueba)
print("Fuentes:", formatear_fuentes(resultados_prueba))
for resultado in resultados_prueba:
    print(f"\n[Página {resultado['pagina']} | puntaje {resultado['puntaje']:.2f}]")
    print(resultado["texto"][:550])

## 8. Reglas del agente y función principal

Solo se envían a Gemini los fragmentos recuperados. Si no hay contexto relevante, Rubi no llama al modelo y responde que no encontró la información.

In [ ]:
SISTEMA = """Sos Rubi, el asistente documental de la empresa ficticia AseguraPy.
Respondé únicamente usando el CONTEXTO RECUPERADO.
No inventes montos, nombres, leyes, direcciones, teléfonos, coberturas, procedimientos ni fechas.
Conservá exactamente montos, límites, plazos y exclusiones.
Si el contexto no permite responder con certeza, respondé exactamente: No encontré esa información en la documentación disponible de AseguraPy.
No confirmes cobertura de casos particulares ni brindes asesoramiento legal, médico o financiero.
Respondé en español claro y breve. Recordá que la empresa es ficticia y el proyecto es académico.
"""

def respuesta_extractiva(resultados):
    if not resultados:
        return "No encontré esa información en la documentación disponible de AseguraPy."
    return ("Gemini no está disponible en este momento. Estos son los fragmentos recuperados del documento:\n\n" +
            "\n\n".join(f"- {r['texto']}" for r in resultados[:2]))

def responder(pregunta):
    pregunta = normalizar(pregunta)
    if not pregunta:
        return "Escribí una consulta para que Rubi pueda buscarla en el documento.", [], "Estado: consulta vacía."

    resultados = buscar_fragmentos(pregunta, k=4)
    if not resultados:
        return "No encontré esa información en la documentación disponible de AseguraPy.", [], "Estado: sin coincidencias documentales."

    contexto = "\n\n".join(f"[Página {r['pagina']}]\n{r['texto']}" for r in resultados)
    fuentes = formatear_fuentes(resultados)
    estado = f"Estado: respuesta basada en {fuentes}."

    if not MODELO_LISTO or client is None:
        return respuesta_extractiva(resultados), resultados, estado + " Modo recuperación local."

    try:
        solicitud = f"CONTEXTO RECUPERADO:\n{contexto}\n\nPREGUNTA DEL USUARIO:\n{pregunta}"
        respuesta = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=solicitud,
            config=types.GenerateContentConfig(system_instruction=SISTEMA, max_output_tokens=450),
        )
        texto = normalizar(getattr(respuesta, "text", ""))
        if not texto:
            texto = "No encontré una respuesta utilizable en la documentación disponible de AseguraPy."
        return texto, resultados, estado
    except Exception as error:
        return respuesta_extractiva(resultados), resultados, estado + " " + explicar_error_api(error)

respuesta, fuentes_prueba, estado_prueba = responder("¿Qué cubre el Plan Ruta por daños a terceros?")
print(respuesta)
print(estado_prueba)

## 9. Interfaz profesional con Gradio 6+

La celda detecta la versión instalada. En Gradio 6, `theme` y `css` se pasan correctamente a `launch()`. También cierra interfaces anteriores antes de abrir una nueva.

In [ ]:
import gradio as gr
print("Versión de Gradio:", gr.__version__)
SOPORTA_MENSAJES = int(gr.__version__.split(".", 1)[0]) >= 6

TEMA = gr.themes.Default(
    primary_hue="red",
    neutral_hue="slate",
    radius_size=gr.themes.sizes.radius_md,
).set(
    body_background_fill="#FFFFFF",
    body_text_color="#111111",
    button_primary_background_fill="#C1121F",
    button_primary_background_fill_hover="#8E0D17",
    button_primary_text_color="#FFFFFF",
)

CSS = """
.gradio-container {max-width: 1050px !important; margin: 0 auto !important; background: #FFFFFF;}
#hero {background: linear-gradient(135deg,#111111 0%,#2A0A0D 55%,#C1121F 150%); color: white; border-radius: 18px; padding: 24px; margin-bottom: 14px;}
#hero h1 {margin: 0; color: #FFFFFF; font-size: 2rem;}
#hero p {margin: 7px 0 0; color: #F5F5F5;}
.badge {display:inline-block; background:#FFFFFF; color:#C1121F; padding:5px 10px; border-radius:999px; font-weight:700; font-size:.82rem; margin-top:12px;}
#status {border-left: 4px solid #C1121F; background:#F5F5F5; padding:10px 13px; border-radius:8px;}
footer {display:none !important;}
@media (max-width: 600px) { #hero {padding:18px;} #hero h1 {font-size:1.55rem;} .gradio-container {padding: 8px !important;} }
"""

BIENVENIDA = "Hola, soy Rubi. Consultá sobre las coberturas, plazos, requisitos y procedimientos de AseguraPy. Responderé solo con información del PDF ficticio."
HISTORIAL_INICIAL = [{"role": "assistant", "content": BIENVENIDA}] if SOPORTA_MENSAJES else [[None, BIENVENIDA]]

def ejecutar_chat(pregunta, historial):
    respuesta, resultados, estado = responder(pregunta)
    fuentes_md = "### Fuentes consultadas\n"
    if resultados:
        fuentes_md += "\n".join(f"- Página {r['pagina']}: {r['texto'][:180]}..." for r in resultados)
    else:
        fuentes_md += "No se encontraron páginas relevantes."
    historial = historial or []
    if SOPORTA_MENSAJES:
        nuevo = historial + [{"role":"user", "content":pregunta}, {"role":"assistant", "content":respuesta}]
    else:
        nuevo = historial + [[pregunta, respuesta]]
    return "", nuevo, fuentes_md, estado

def limpiar_chat():
    return "", HISTORIAL_INICIAL, "### Fuentes consultadas\nAún no hay una consulta.", "Estado: documento procesado; esperando consulta."

try:
    gr.close_all()
except Exception:
    pass

with gr.Blocks() as demo:
    gr.HTML("""<section id='hero'><h1>◈ Rubi | AseguraPy</h1><p>Agente documental de seguros · Proyecto académico ficticio</p><span class='badge'>Respuestas basadas en fuentes</span></section>""")
    estado = gr.Markdown(f"<div id='status'>Estado: PDF procesado ({len(paginas)} páginas, {len(fragmentos)} fragmentos).</div>")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot_kwargs = {"label": "Conversación con Rubi", "height": 440}
            if SOPORTA_MENSAJES:
                chatbot_kwargs["type"] = "messages"
            chat = gr.Chatbot(value=HISTORIAL_INICIAL, **chatbot_kwargs)
            with gr.Row():
                entrada = gr.Textbox(label="Tu consulta", placeholder="Ej.: ¿Cuál es el plazo para denunciar un robo de vehículo?", scale=5)
                enviar = gr.Button("Enviar", variant="primary", scale=1)
            limpiar = gr.Button("Limpiar conversación")
        with gr.Column(scale=2):
            fuentes = gr.Markdown("### Fuentes consultadas\nAún no hay una consulta.")
            gr.Markdown("### Preguntas rápidas")
            sugeridas = [
                "¿Cuál es el límite por daños a terceros del Plan Ruta?",
                "¿Qué documentos necesito para denunciar un choque?",
                "¿Cuánto tarda la revisión inicial?",
                "¿Casa Serena cubre humedad gradual?",
                "¿Qué datos no debo enviar por el chat?",
                "¿Cómo funciona el estado Documentación pendiente?",
            ]
            botones = [gr.Button(s) for s in sugeridas]
    gr.Markdown("---\n**AseguraPy es una empresa ficticia.** Rubi no confirma coberturas individuales ni reemplaza atención profesional.")

    eventos = dict(inputs=[entrada, chat], outputs=[entrada, chat, fuentes, estado])
    enviar.click(ejecutar_chat, **eventos)
    entrada.submit(ejecutar_chat, **eventos)
    limpiar.click(limpiar_chat, outputs=[entrada, chat, fuentes, estado])
    for boton, texto in zip(botones, sugeridas):
        boton.click(lambda consulta=texto: ejecutar_chat(consulta, []), outputs=[entrada, chat, fuentes, estado])

# Prueba en Colab. En OCI se cambiará share=False y se usarán server_name y server_port.
demo.launch(share=True, debug=True, theme=TEMA, css=CSS)

## 10. Pruebas recomendadas y errores frecuentes

Probá: monto (daños a terceros), plazo (robo vehicular), seguridad, procedimiento, seguimiento, una pregunta sin información (teléfono), una ambigua y una vacía.

- **PDF sin texto:** reemplazarlo por un PDF que permita seleccionar texto.
- **401:** revisar el secreto `GEMINI_API_KEY`, sin mostrar ni copiar la clave.
- **404:** ejecutar `listar_modelos_disponibles()` y elegir un modelo de la lista.
- **429:** esperar el reinicio de cuota o seguir en modo BM25 local.
- **Gradio no abre:** volver a ejecutar la última celda; `gr.close_all()` cierra demos anteriores.
- **Respuesta fuera del PDF:** revisar los fragmentos y reducir `k` o `max_output_tokens`; no relajar el prompt de seguridad.